# Adopción de inteligencia artificial en las empresas españolas (2022–2026)

**Análisis exploratorio de datos (EDA).** Evolución de la adopción de IA, diferencias por tamaño y sector, tecnologías, usos, barreras y relación exploratoria con la productividad laboral.

**Fuentes:** INE (Encuesta sobre el uso de TIC y comercio electrónico en las empresas) y Eurostat (`isoc_eb_ai`, `nama_10_a64`, `nama_10_a64_e`).

> **Cómo ejecutar:** abrir el notebook desde `src/` o desde la raíz del proyecto, instalar las dependencias y ejecutar las celdas en orden. Los CSV originales deben encontrarse en `src/data/raw/`. Las comparaciones entre años requieren cautela por posibles cambios metodológicos y por la clasificación CNAE de 2026.

## 1. Preparación y calidad de los datos

In [ ]:
# Librerías y rutas del proyecto
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

BASE = Path.cwd()
SRC_PATH = BASE if (BASE / "data" / "raw").exists() else BASE / "src"
DATA_PATH = SRC_PATH / "data" / "raw"
OUTPUT_PATH = SRC_PATH / "data" / "processed"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Solo los cinco CSV del INE; no mezclar con descargas de Eurostat.
archivos_ine = [DATA_PATH / f"Implantacion IA empresas ({year}).csv" for year in range(2022, 2027)]
no_encontrados = [str(p) for p in archivos_ine if not p.exists()]
if no_encontrados:
    raise FileNotFoundError(f"Faltan archivos INE: {no_encontrados}")

dataframes = []
for year, archivo in zip(range(2022, 2027), archivos_ine):
    temporal = pd.read_csv(archivo, sep=";", encoding="utf-8-sig", decimal=",")
    temporal["Año"] = year
    dataframes.append(temporal)

df_original = pd.concat(dataframes, ignore_index=True)
print("Registros por año:")
display(df_original.groupby("Año").size().to_frame("Registros"))
print("Dimensiones:", df_original.shape)
print("Nulos por columna (los de clasificación sectorial pueden ser estructurales):")
display(df_original.isna().sum().to_frame("Nulos"))
print("Duplicados exactos:", df_original.duplicated().sum())
print("Porcentajes fuera de 0–100:", (~df_original["Total"].between(0, 100)).sum())

In [ ]:
# Unificar la denominación sectorial de los CSV de 2022–2025 y 2026
columnas_sector = [
    "Agrupación de actividad (excepto CNAE 56, 64-66 y 95.1)",
    "Agrupación de actividad (CNAE-25 sección C-O, excl. L y grupo 951)"
]
df_original["sector_raw"] = df_original[columnas_sector[0]].combine_first(
    df_original[columnas_sector[1]]
)
print("Sectores sin clasificar:", df_original["sector_raw"].isna().sum())
display(df_original.groupby("Año")["sector_raw"].nunique().to_frame("Sectores distintos"))

In [ ]:
print("Duplicados exactos:", df_original.duplicated().sum())

# Comprobar duplicados por combinación de variables
claves = [
    "Año",
    "sector_raw",
    "Principales variables",
    "Tamaño de la empresa"
]

print(
    "Duplicados por combinación de año, sector, indicador y tamaño:",
    df_original.duplicated(subset=claves).sum()
)

In [ ]:
# Crear dataset de trabajo
df = df_original[
    [
        "Año",
        "sector_raw",
        "Principales variables",
        "Tamaño de la empresa",
        "Total"
    ]
].copy()

# Renombrar columnas
df = df.rename(columns={
    "Año": "year",
    "sector_raw": "sector",
    "Principales variables": "indicator",
    "Tamaño de la empresa": "company_size",
    "Total": "value_pct"
})

# Ordenar registros
df = df.sort_values(
    ["year", "sector", "indicator", "company_size"]
).reset_index(drop=True)

# Comprobar estructura
print("Dimensiones:", df.shape)
print("\nColumnas:", df.columns.tolist())
display(df.head())

# Guardar dataset procesado
df.to_csv(
    OUTPUT_PATH / "dataset_ia_empresas_2022_2026.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nDataset guardado correctamente.")

**Control metodológico:** las columnas sectoriales originales cambian en 2026. La media de `value_pct` sobre indicadores heterogéneos no representa una tasa general de adopción. Para cada hipótesis se selecciona su indicador específico.

## 2. H1 — Evolución de la adopción de IA en España

In [ ]:
# Filtrar el indicador general de adopción de IA
mascara = df["indicator"].str.contains(
    "emplean tecnologías de Inteligencia Artificial",
    case=False,
    na=False
)

df_adopcion = df.loc[mascara].copy()

# Mostrar los registros de Total Empresas y Total
df_h1 = df_adopcion[
    (df_adopcion["sector"].str.contains(
        "Total Empresas", case=False, na=False
    )) &
    (df_adopcion["company_size"] == "Total")
].copy()

# Mostrar resultados
display(
    df_h1[["year", "indicator", "value_pct"]]
    .sort_values("year")
    .reset_index(drop=True)
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df_h1 = df_h1.sort_values("year").copy()

plt.figure(figsize=(10, 5))

sns.lineplot(
    data=df_h1,
    x="year",
    y="value_pct",
    marker="o",
    linewidth=2.5,
    color="#2563EB"
)

for _, fila in df_h1.iterrows():
    plt.annotate(
        f"{fila['value_pct']:.2f}%",
        (fila["year"], fila["value_pct"]),
        xytext=(0, 10),
        textcoords="offset points",
        ha="center"
    )

plt.title("Evolución de la adopción de IA en empresas españolas")
plt.xlabel("Año")
plt.ylabel("Empresas que utilizan IA (%)")
plt.xticks(df_h1["year"])
plt.ylim(0, 40)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

**Hallazgo H1.** La adopción general pasa del **12,60 % (2022)** al **33,60 % (2026)**, un aumento de **21 puntos porcentuales**. No crece de forma monótona: en 2023 se observa una caída. La comparación temporal debe interpretarse con cautela por posibles cambios de cobertura y metodología.

### 2.1. Comparación con la Unión Europea (Eurostat, 2023–2025)

In [ ]:
# Serie histórica oficial Eurostat (España y UE-27)
df_evolucion_ue = pd.read_csv(DATA_PATH / "evolucion_UE.csv")
print("Registros descargados:", df_evolucion_ue.shape)

In [ ]:
# H1.1 - Serie histórica oficial España vs UE

df_europa_oficial = df_evolucion_ue[
    (df_evolucion_ue["indic_is"] == "E_AI_TANY") &
    (df_evolucion_ue["geo"].isin(["ES", "EU27_2020"])) &
    (df_evolucion_ue["size_emp"] == "GE10") &
    (df_evolucion_ue["unit"] == "PC_ENT") &
    (df_evolucion_ue["nace_r2"] == "C10-S951_X_K")
].copy()

# Convertir los porcentajes a valores numéricos
df_europa_oficial["OBS_VALUE"] = pd.to_numeric(
    df_europa_oficial["OBS_VALUE"],
    errors="coerce"
)

# Crear tabla comparativa
tabla_europa = df_europa_oficial.pivot(
    index="TIME_PERIOD",
    columns="geo",
    values="OBS_VALUE"
)

tabla_europa = tabla_europa.rename(columns={
    "ES": "España",
    "EU27_2020": "UE-27"
})

display(tabla_europa)

In [ ]:
# Actualizar la serie europea con los datos oficiales

df_europa = (
    tabla_europa
    .reset_index()
    .rename(columns={"TIME_PERIOD": "year"})
)

display(df_europa)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(
    df_europa["year"],
    df_europa["España"],
    marker="o",
    linewidth=2.5,
    color="#2563EB",
    label="España"
)

ax.plot(
    df_europa["year"],
    df_europa["UE-27"],
    marker="o",
    linewidth=2.5,
    color="#F59E0B",
    label="Unión Europea"
)

# Mostrar porcentajes sin solapamientos
for año, espana, ue in zip(
    df_europa["year"],
    df_europa["España"],
    df_europa["UE-27"]
):
    if espana > ue:
        desplazamiento_es = (0, 15)
        desplazamiento_ue = (0, -25)
    else:
        desplazamiento_es = (0, -25)
        desplazamiento_ue = (0, 15)

    ax.annotate(
        f"{espana:.2f}%",
        (año, espana),
        xytext=desplazamiento_es,
        textcoords="offset points",
        ha="center",
        color="#2563EB",
        fontsize=10
    )

    ax.annotate(
        f"{ue:.2f}%",
        (año, ue),
        xytext=desplazamiento_ue,
        textcoords="offset points",
        ha="center",
        color="#F59E0B",
        fontsize=10
    )

ax.set_title(
    "Evolución de la adopción de IA: España vs UE (2023-2025)",
    fontsize=14,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Año")
ax.set_ylabel("Empresas que utilizan IA (%)")
ax.set_xticks(df_europa["year"])
ax.set_ylim(0, 25)

ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
# H1.1 - Crecimiento de la adopción de IA en España y la UE

crecimiento_espana = (
    df_europa.loc[df_europa["year"] == 2025, "España"].iloc[0]
    - df_europa.loc[df_europa["year"] == 2023, "España"].iloc[0]
)

crecimiento_ue = (
    df_europa.loc[df_europa["year"] == 2025, "UE-27"].iloc[0]
    - df_europa.loc[df_europa["year"] == 2023, "UE-27"].iloc[0]
)

print(f"Crecimiento España: +{crecimiento_espana:.1f} puntos porcentuales")
print(f"Crecimiento UE: +{crecimiento_ue:.1f} puntos porcentuales")
print(f"Diferencia de crecimiento: {crecimiento_espana - crecimiento_ue:.1f} puntos")

In [ ]:
# H1.1 - Crecimiento anual de adopción de IA

crecimiento_anual = df_europa.copy()

crecimiento_anual["crecimiento_espana_pp"] = (
    crecimiento_anual["España"].diff()
)

crecimiento_anual["crecimiento_ue_pp"] = (
    crecimiento_anual["UE-27"].diff()
)

display(crecimiento_anual.round(2))

**Hallazgo europeo.** En 2025 España alcanza **20,27 %** y la UE-27 **19,95 %** en la serie comparable de Eurostat. No se mezclan los porcentajes nacionales del INE con los de Eurostat en una misma serie temporal.

### 2.2. Posición de España en la UE-27 (2025)

In [ ]:
# Ranking de adopción de IA en la UE-27 (Eurostat)
df_ue = pd.read_csv(DATA_PATH / "datosUE.csv")
print("Dimensiones:", df_ue.shape)

In [ ]:
# H1.2 - Filtrar adopción general de IA en 2025

df_ranking_ue = df_ue[
    (df_ue["indic_is"] == "E_AI_TANY") &
    (df_ue["TIME_PERIOD"] == 2025) &
    (df_ue["size_emp"] == "GE10") &
    (df_ue["unit"] == "PC_ENT")
].copy()

# Seleccionar las columnas necesarias
df_ranking_ue = df_ranking_ue[
    ["geo", "Geopolitical entity (reporting)", "OBS_VALUE"]
]

# Renombrar columnas
df_ranking_ue = df_ranking_ue.rename(columns={
    "Geopolitical entity (reporting)": "pais",
    "OBS_VALUE": "adopcion_ia_pct"
})

# Ordenar de mayor a menor adopción
df_ranking_ue = df_ranking_ue.sort_values(
    "adopcion_ia_pct",
    ascending=False
)

print("Territorios encontrados:", len(df_ranking_ue))

display(df_ranking_ue.reset_index(drop=True))

In [ ]:
# H1.2 - Ranking de los 27 países de la Unión Europea

paises_ue27 = [
    "AT", "BE", "BG", "HR", "CY", "CZ", "DK",
    "EE", "FI", "FR", "DE", "EL", "HU", "IE",
    "IT", "LV", "LT", "LU", "MT", "NL", "PL",
    "PT", "RO", "SK", "SI", "ES", "SE"
]

ranking_ue27 = (
    df_ranking_ue[
        df_ranking_ue["geo"].isin(paises_ue27)
    ]
    .sort_values("adopcion_ia_pct", ascending=False)
    .reset_index(drop=True)
)

ranking_ue27.index = ranking_ue27.index + 1

print("Países de la UE encontrados:", len(ranking_ue27))

print("\nPosición de España:")
display(ranking_ue27[ranking_ue27["geo"] == "ES"])

print("\nRanking completo:")
display(ranking_ue27)

In [ ]:
# H1.2 - Ranking europeo de adopción de IA (2025)

import matplotlib.pyplot as plt

grafico_ue = ranking_ue27.reset_index(drop=True).copy()

# Colores: España destacada y resto de países en azul
colores = [
    "#F59E0B" if codigo == "ES" else "#2563EB"
    for codigo in grafico_ue["geo"]
]

# Obtener el agregado UE-27 del CSV de Eurostat
media_ue = df_ranking_ue.loc[
    df_ranking_ue["geo"] == "EU27_2020",
    "adopcion_ia_pct"
].iloc[0]

fig, ax = plt.subplots(figsize=(12, 11))

ax.barh(
    grafico_ue["pais"],
    grafico_ue["adopcion_ia_pct"],
    color=colores
)

# País con mayor adopción arriba
ax.invert_yaxis()

# Mostrar porcentajes al final de cada barra
for i, valor in enumerate(grafico_ue["adopcion_ia_pct"]):
   ax.text(
    valor + 0.4,
    i,
    f"{valor:.2f}%",
    va="center",
    fontsize=9,
    bbox=dict(
        facecolor="white",
        edgecolor="none",
        alpha=0.85,
        pad=0.5
    )
)

# Línea de referencia de la UE
ax.axvline(
    media_ue,
    color="#DC2626",
    linestyle="--",
    linewidth=2,
    label=f"UE-27: {media_ue:.2f}%"
)

ax.set_title(
    "Adopción de inteligencia artificial en empresas de la UE (2025)",
    fontsize=14,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Empresas que utilizan IA (%)")
ax.set_ylabel("")
ax.set_xlim(0, 48)

ax.legend(loc="lower right")
ax.grid(axis="x", alpha=0.2)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

**Hallazgo del ranking.** España ocupa el **puesto 13 de 27** en 2025. La media agregada UE-27 es una estadística ponderada y no debe confundirse con la media aritmética de países.

## 3. H2 — Diferencias de adopción según el tamaño empresarial

In [ ]:
# Hipótesis H2: Las empresas grandes adoptan inteligencia artificial en mayor proporción que las pequeñas.

# Filtrar adopción general para el conjunto de empresas
df_h2 = df_adopcion[
    (df_adopcion["sector"] == "Total Empresas") &
    (df_adopcion["company_size"] != "Total")
].copy()

# Establecer el orden de los tamaños
orden_tamanos = [
    "De 10 a 49",
    "De 50 a 249",
    "De 250 y más"
]

df_h2["company_size"] = pd.Categorical(
    df_h2["company_size"],
    categories=orden_tamanos,
    ordered=True
)

# Tabla comparativa
tabla_h2 = df_h2.pivot(
    index="year",
    columns="company_size",
    values="value_pct"
)

display(tabla_h2.round(2))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(11, 6))

sns.barplot(
    data=df_h2,
    x="year",
    y="value_pct",
    hue="company_size",
    hue_order=[
        "De 10 a 49",
        "De 50 a 249",
        "De 250 y más"
    ],
    palette="Blues"
)

plt.title(
    "Adopción de inteligencia artificial según el tamaño de empresa",
    fontsize=14,
    fontweight="bold"
)
plt.xlabel("Año")
plt.ylabel("Empresas que utilizan IA (%)")
plt.ylim(0, 85)
plt.legend(title="Tamaño de empresa")
plt.grid(axis="y", alpha=0.2)

plt.tight_layout()
plt.show()

**Hallazgo H2.** Las grandes empresas superan a las pequeñas en todos los años observados. En 2026 registran **74,39 %**, frente al **29,14 %** de las pequeñas: una brecha de **45,25 puntos porcentuales**.

## 4. H3 — Diferencias de adopción según el sector económico (2025)

In [ ]:
## H3. Adopción de inteligencia artificial por sector económico

# Seleccionar el año 2025 y el tamaño Total
df_h3 = df_adopcion[
    (df_adopcion["year"] == 2025) &
    (df_adopcion["company_size"] == "Total") &
    (df_adopcion["sector"] != "Total Empresas")
].copy()

# Ordenar sectores según su adopción de IA
df_h3 = df_h3.sort_values(
    "value_pct",
    ascending=False
)

print("Número de sectores:", len(df_h3))

display(
    df_h3[["sector", "value_pct"]]
    .reset_index(drop=True)
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Crear una copia de los datos
df_h3_grafico = df_h3.copy()

# Nombres abreviados siguiendo el orden de los sectores por porcentaje
nombres_cortos = [
    "Información y comunicaciones",
    "Sector TIC",
    "Actividades profesionales y científicas",
    "Actividades inmobiliarias",
    "Total servicios",
    "Energía y agua",
    "Alojamiento",
    "Fabricación de productos informáticos",
    "Coquerías y refino de petróleo",
    "Servicios administrativos",
    "Comercio y reparación de vehículos",
    "Total industria",
    "Alimentación, textil y otras manufacturas",
    "Transporte y almacenamiento",
    "Metalurgia y productos metálicos",
    "Total construcción"
]

# Ordenar de mayor a menor
df_h3_grafico = df_h3_grafico.sort_values(
    "value_pct", ascending=False
).reset_index(drop=True)

df_h3_grafico["sector_corto"] = nombres_cortos

# Crear gráfico
fig, ax = plt.subplots(figsize=(12, 8))

sns.barplot(
    data=df_h3_grafico,
    x="value_pct",
    y="sector_corto",
    color="#2563EB",
    ax=ax
)

# Añadir porcentajes
for barra in ax.patches:
    valor = barra.get_width()

    ax.text(
        valor + 0.5,
        barra.get_y() + barra.get_height() / 2,
        f"{valor:.2f}%",
        va="center",
        fontsize=9
    )

ax.set_title(
    "Adopción de IA por sector económico en España (2025)",
    fontsize=14,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Empresas que utilizan IA (%)")
ax.set_ylabel("")
ax.set_xlim(0, 70)
ax.grid(axis="x", alpha=0.2)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# H3 - Análisis complementario:
# Adopción de IA en pequeñas empresas por sector (2025)

df_pequenas = df_adopcion[
    (df_adopcion["year"] == 2025) &
    (df_adopcion["company_size"] == "De 10 a 49") &
    (df_adopcion["sector"] != "Total Empresas")
].copy()

df_pequenas = df_pequenas.sort_values(
    "value_pct",
    ascending=False
)

display(
    df_pequenas[
        ["sector", "value_pct"]
    ].reset_index(drop=True)
)

In [ ]:
# H3 - Análisis complementario:
# Brecha de adopción entre grandes y pequeñas empresas (2025)

df_brechas = df_adopcion[
    (df_adopcion["year"] == 2025) &
    (df_adopcion["sector"] != "Total Empresas") &
    (
        df_adopcion["company_size"].isin([
            "De 10 a 49",
            "De 250 y más"
        ])
    )
].copy()

# Crear tabla comparativa por sector
tabla_brechas = df_brechas.pivot(
    index="sector",
    columns="company_size",
    values="value_pct"
)

# Calcular la brecha en puntos porcentuales
tabla_brechas["brecha_pp"] = (
    tabla_brechas["De 250 y más"]
    - tabla_brechas["De 10 a 49"]
)

# Ordenar de mayor a menor brecha
tabla_brechas = tabla_brechas.sort_values(
    "brecha_pp",
    ascending=False
)

display(tabla_brechas.round(2))

In [ ]:
# Gráfico de brechas sectoriales de adopción de IA (2025)

import matplotlib.pyplot as plt

etiquetas_brechas = [
    "Coquerías, refino, farmacéutica y otros",
    "Metalurgia y productos metálicos",
    "Maquinaria, equipos y otras manufacturas",
    "Total industria",
    "Alimentación, textil y otras manufacturas",
    "Actividades profesionales y científicas",
    "Comercio y reparación de vehículos",
    "Total construcción",
    "Energía y agua",
    "Transporte y almacenamiento",
    "Alojamiento",
    "Total servicios",
    "Información y comunicaciones",
    "Sector TIC",
    "Actividades inmobiliarias",
    "Servicios administrativos y auxiliares"
]

grafico_brechas = tabla_brechas.reset_index().copy()
grafico_brechas["sector_corto"] = etiquetas_brechas

fig, ax = plt.subplots(figsize=(12, 8))

ax.barh(
    grafico_brechas["sector_corto"],
    grafico_brechas["brecha_pp"],
    color="#2563EB"
)

ax.invert_yaxis()

for i, valor in enumerate(grafico_brechas["brecha_pp"]):
    ax.text(
        valor + 0.5,
        i,
        f"{valor:.2f}",
        va="center",
        fontsize=9
    )

ax.set_title(
    "Brecha de adopción de IA entre empresas grandes y pequeñas (2025)",
    fontsize=14,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Diferencia en puntos porcentuales")
ax.set_ylabel("")
ax.set_xlim(0, 65)
ax.grid(axis="x", alpha=0.2)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# H2 + H3 - Mapa de calor de adopción de IA (2025)

import matplotlib.pyplot as plt
import seaborn as sns

# Filtrar datos de 2025
df_heatmap = df_adopcion[
    (df_adopcion["year"] == 2025) &
    (df_adopcion["sector"] != "Total Empresas") &
    (df_adopcion["company_size"] != "Total")
].copy()

# Crear tabla para el mapa de calor
tabla_heatmap = df_heatmap.pivot(
    index="sector",
    columns="company_size",
    values="value_pct"
)

# Ordenar los tamaños de menor a mayor
tabla_heatmap = tabla_heatmap[
    ["De 10 a 49", "De 50 a 249", "De 250 y más"]
]

# Utilizar los nombres cortos de sectores que ya creamos
mapa_nombres = dict(
    zip(df_h3_grafico["sector"], df_h3_grafico["sector_corto"])
)

tabla_heatmap = tabla_heatmap.rename(index=mapa_nombres)

# Ordenar sectores según adopción en pequeñas empresas
tabla_heatmap = tabla_heatmap.sort_values(
    "De 10 a 49",
    ascending=False
)

# Dibujar mapa de calor
fig, ax = plt.subplots(figsize=(11, 9))

sns.heatmap(
    tabla_heatmap,
    annot=True,
    fmt=".1f",
    cmap="Blues",
    linewidths=0.5,
    vmin=0,
    vmax=100,
    cbar_kws={"label": "Empresas que utilizan IA (%)"},
    ax=ax
)

ax.set_title(
    "Adopción de IA por sector y tamaño empresarial (2025)",
    fontsize=14,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Tamaño de empresa")
ax.set_ylabel("")
ax.set_xticklabels(
    ["Pequeñas (10-49)", "Medianas (50-249)", "Grandes (250+)"],
    rotation=0
)
ax.tick_params(axis="y", labelrotation=0)

plt.tight_layout()
plt.show()

**Hallazgo H3.** Información y comunicaciones registra **60,57 %** de adopción en 2025, frente al **11,44 %** de construcción. El tamaño empresarial también marca diferencias dentro de cada sector. Las categorías sectoriales incluyen algunos agregados y grupos transversales; no deben sumarse entre sí.

## 5. H4 — Tecnologías, aplicaciones y barreras de adopción (2025)

**Precaución sobre denominadores:** las notas específicas `(2)` y `(3)` de los indicadores no han sido verificadas. No interpretar automáticamente estos porcentajes como proporciones del total de empresas ni comparar magnitudes de bloques distintos como si compartieran denominador.

### 5.1. Tecnologías utilizadas

In [ ]:
# H4.1 - Tecnologías de IA utilizadas por las empresas (2025)

df_h4_tecnologias = df[
    (df["year"] == 2025) &
    (df["sector"] == "Total Empresas") &
    (df["company_size"] == "Total") &
    (df["indicator"].str.match(r"^E\.1\.[A-H]\s"))
].copy()

# Extraer código y descripción
df_h4_tecnologias["codigo"] = (
    df_h4_tecnologias["indicator"]
    .str.extract(r"^(E\.1\.[A-H])")
)

df_h4_tecnologias = df_h4_tecnologias.sort_values(
    "value_pct", ascending=False
)

display(
    df_h4_tecnologias[
        ["codigo", "indicator", "value_pct"]
    ]
)

In [ ]:
# H4.1 - Gráfico de tecnologías de IA utilizadas en 2025

nombres_tecnologias = {
    "E.1.A": "Análisis de texto",
    "E.1.B": "Conversión de voz a texto",
    "E.1.C": "Generación de texto, voz o código",
    "E.1.D": "Generación de imágenes, vídeo o audio",
    "E.1.E": "Reconocimiento de objetos o personas",
    "E.1.F": "Machine learning para análisis de datos",
    "E.1.G": "Automatización de procesos y decisiones",
    "E.1.H": "Movimiento autónomo de máquinas"
}

df_h4_grafico = df_h4_tecnologias.copy()

df_h4_grafico["tecnologia"] = (
    df_h4_grafico["codigo"].map(nombres_tecnologias)
)

df_h4_grafico = df_h4_grafico.sort_values(
    "value_pct", ascending=True
)

fig, ax = plt.subplots(figsize=(11, 6))

colores = [
    "#F59E0B" if valor == df_h4_grafico["value_pct"].max()
    else "#2563EB"
    for valor in df_h4_grafico["value_pct"]
]

ax.barh(
    df_h4_grafico["tecnologia"],
    df_h4_grafico["value_pct"],
    color=colores
)

for i, valor in enumerate(df_h4_grafico["value_pct"]):
    ax.text(
        valor + 0.7,
        i,
        f"{valor:.2f}%",
        va="center",
        fontsize=10
    )

ax.set_title(
    "Tecnologías de IA utilizadas por las empresas españolas (2025)",
    fontsize=14,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Porcentaje (%)")
ax.set_ylabel("")
ax.set_xlim(0, 65)
ax.grid(axis="x", alpha=0.2)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

**Hallazgo.** Destacan la generación de imágenes, vídeo o audio (**53,87 %**) y el análisis de texto (**49,55 %**), dentro de la población de referencia del indicador.

### 5.2. Aplicaciones empresariales

In [ ]:
# H4.2 - Aplicaciones empresariales de IA (2025)

df_h4_usos = df[
    (df["year"] == 2025) &
    (df["sector"] == "Total Empresas") &
    (df["company_size"] == "Total") &
    (df["indicator"].str.match(r"^E\.2\.[A-G]\s"))
].copy()

df_h4_usos["codigo"] = (
    df_h4_usos["indicator"]
    .str.extract(r"^(E\.2\.[A-G])")
)

df_h4_usos = df_h4_usos.sort_values(
    "value_pct",
    ascending=False
)

display(
    df_h4_usos[
        ["codigo", "indicator", "value_pct"]
    ]
)

In [ ]:
# H4.2 - Gráfico de aplicaciones empresariales de IA (2025)

nombres_usos = {
    "E.2.A": "Marketing y ventas",
    "E.2.B": "Producción de bienes o servicios",
    "E.2.C": "Administración y gestión empresarial",
    "E.2.D": "Logística",
    "E.2.E": "Seguridad de las TIC",
    "E.2.F": "Gestión contable y financiera",
    "E.2.G": "Investigación, desarrollo e innovación"
}

df_h4_usos_grafico = df_h4_usos.copy()

df_h4_usos_grafico["uso"] = (
    df_h4_usos_grafico["codigo"].map(nombres_usos)
)

df_h4_usos_grafico = df_h4_usos_grafico.sort_values(
    "value_pct",
    ascending=True
)

fig, ax = plt.subplots(figsize=(11, 6))

colores = [
    "#F59E0B" if valor == df_h4_usos_grafico["value_pct"].max()
    else "#2563EB"
    for valor in df_h4_usos_grafico["value_pct"]
]

ax.barh(
    df_h4_usos_grafico["uso"],
    df_h4_usos_grafico["value_pct"],
    color=colores
)

for i, valor in enumerate(df_h4_usos_grafico["value_pct"]):
    ax.text(
        valor + 0.5,
        i,
        f"{valor:.2f}%",
        va="center",
        fontsize=10
    )

ax.set_title(
    "Principales aplicaciones empresariales de IA en España (2025)",
    fontsize=14,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Porcentaje (%)")
ax.set_ylabel("")
ax.set_xlim(0, 37)
ax.grid(axis="x", alpha=0.2)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

**Hallazgo.** Los mayores porcentajes corresponden a administración y gestión (**29,75 %**) y marketing y ventas (**29,61 %**).

### 5.3. Comparación exploratoria entre tecnología multimedia y marketing por sector

In [ ]:
# H4 - Cruce entre IA generativa y marketing por sector (2025)

df_h4_cruce = df[
    (df["year"] == 2025) &
    (df["company_size"] == "Total") &
    (df["sector"] != "Total Empresas") &
    (
        df["indicator"].str.match(r"^E\.1\.D\s") |
        df["indicator"].str.match(r"^E\.2\.A\s")
    )
].copy()

# Identificar los dos indicadores
df_h4_cruce["tipo"] = df_h4_cruce["indicator"].apply(
    lambda x: "IA generativa multimedia"
    if x.startswith("E.1.D")
    else "Marketing y ventas"
)

# Crear tabla por sector
tabla_h4_cruce = df_h4_cruce.pivot(
    index="sector",
    columns="tipo",
    values="value_pct"
)

# Ordenar por utilización de IA generativa
tabla_h4_cruce = tabla_h4_cruce.sort_values(
    "IA generativa multimedia",
    ascending=False
)

display(tabla_h4_cruce.round(2))

In [ ]:
# H4 - Correlación sectorial entre IA generativa multimedia y marketing

# Excluir agregados y sector TIC transversal
sectores_excluir = [
    "1. Total Industria",
    "2. Total Construcción",
    "3. Total Servicios",
    "4. Sector TIC"
]

tabla_correlacion = tabla_h4_cruce[
    ~tabla_h4_cruce.index.to_series().str.startswith(
        tuple(sectores_excluir)
    )
].copy()

# Calcular correlación de Pearson
correlacion = tabla_correlacion[
    "IA generativa multimedia"
].corr(
    tabla_correlacion["Marketing y ventas"]
)

print(f"Categorías analizadas: {len(tabla_correlacion)}")
print(f"Correlación de Pearson: {correlacion:.3f}")

In [ ]:
# H4 - Gráfico de dispersión: IA multimedia y marketing

fig, ax = plt.subplots(figsize=(10, 7))

sns.regplot(
    data=tabla_correlacion,
    x="IA generativa multimedia",
    y="Marketing y ventas",
    color="#2563EB",
    scatter_kws={"s": 85, "alpha": 0.8},
    line_kws={"color": "#F59E0B", "linewidth": 2},
    ci=None,
    ax=ax
)

ax.set_title(
    "Relación sectorial entre IA multimedia y marketing (2025)",
    fontsize=14,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Empresas usuarias de IA multimedia (%)")
ax.set_ylabel("Empresas que utilizan IA para marketing (%)")

ax.text(
    0.05, 0.95,
    f"Pearson r = {correlacion:.3f}",
    transform=ax.transAxes,
    fontsize=12,
    fontweight="bold",
    va="top"
)

ax.grid(alpha=0.2)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

**Hallazgo.** La correlación sectorial entre los dos indicadores es cercana a cero (**r ≈ −0,041**). No se dispone de microdatos para saber si las mismas empresas utilizan ambas opciones: es una comparación de porcentajes agregados, no un cruce individual.

### 5.4. Barreras de adopción

In [ ]:
# H4.3 - Barreras para la adopción de IA (2025)

df_h4_barreras = df[
    (df["year"] == 2025) &
    (df["sector"] == "Total Empresas") &
    (df["company_size"] == "Total") &
    (df["indicator"].str.match(r"^E\.5\.[A-H]\s"))
].copy()

# Extraer código de cada barrera
df_h4_barreras["codigo"] = (
    df_h4_barreras["indicator"]
    .str.extract(r"^(E\.5\.[A-H])")
)

# Ordenar de mayor a menor porcentaje
df_h4_barreras = df_h4_barreras.sort_values(
    "value_pct",
    ascending=False
)

display(
    df_h4_barreras[
        ["codigo", "indicator", "value_pct"]
    ]
)

In [ ]:
# H4.3 - Gráfico de barreras para adoptar IA (2025)

nombres_barreras = {
    "E.5.A": "Costes demasiado elevados",
    "E.5.B": "Falta de conocimientos especializados",
    "E.5.C": "Falta de disponibilidad o calidad de datos",
    "E.5.D": "Incompatibilidad con sistemas existentes",
    "E.5.E": "Protección de datos y privacidad",
    "E.5.F": "Incertidumbre legal",
    "E.5.G": "Consideraciones éticas",
    "E.5.H": "La IA no resulta útil para la empresa"
}

df_h4_barreras_grafico = df_h4_barreras.copy()

df_h4_barreras_grafico["barrera"] = (
    df_h4_barreras_grafico["codigo"].map(nombres_barreras)
)

df_h4_barreras_grafico = df_h4_barreras_grafico.sort_values(
    "value_pct",
    ascending=True
)

fig, ax = plt.subplots(figsize=(11, 6))

colores = [
    "#F59E0B" if valor == df_h4_barreras_grafico["value_pct"].max()
    else "#2563EB"
    for valor in df_h4_barreras_grafico["value_pct"]
]

ax.barh(
    df_h4_barreras_grafico["barrera"],
    df_h4_barreras_grafico["value_pct"],
    color=colores
)

for i, valor in enumerate(df_h4_barreras_grafico["value_pct"]):
    ax.text(
        valor + 0.7,
        i,
        f"{valor:.2f}%",
        va="center",
        fontsize=10
    )

ax.set_title(
    "Principales barreras para la adopción de IA en España (2025)",
    fontsize=14,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Porcentaje (%)")
ax.set_ylabel("")
ax.set_xlim(0, 95)
ax.grid(axis="x", alpha=0.2)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# H4.4 - Barreras de adopción de IA según tamaño empresarial (2025)

df_barreras_tamano = df[
    (df["year"] == 2025) &
    (df["sector"] == "Total Empresas") &
    (df["company_size"] != "Total") &
    (df["indicator"].str.match(r"^E\.5\.[A-H]\s"))
].copy()

# Extraer código de la barrera
df_barreras_tamano["codigo"] = (
    df_barreras_tamano["indicator"]
    .str.extract(r"^(E\.5\.[A-H])")
)

# Asignar nombres cortos
df_barreras_tamano["barrera"] = (
    df_barreras_tamano["codigo"].map(nombres_barreras)
)

# Crear tabla cruzada
tabla_barreras_tamano = df_barreras_tamano.pivot(
    index="barrera",
    columns="company_size",
    values="value_pct"
)

# Ordenar tamaños de menor a mayor
tabla_barreras_tamano = tabla_barreras_tamano[
    ["De 10 a 49", "De 50 a 249", "De 250 y más"]
]

# Ordenar por el porcentaje de pequeñas empresas
tabla_barreras_tamano = tabla_barreras_tamano.sort_values(
    "De 10 a 49",
    ascending=False
)

display(tabla_barreras_tamano.round(2))

In [ ]:
# H4.4 - Comparación visual de barreras según tamaño empresarial

fig, ax = plt.subplots(figsize=(12, 7))

tamanos = [
    "De 10 a 49",
    "De 50 a 249",
    "De 250 y más"
]

colores = ["#2563EB", "#F59E0B", "#16A34A"]
etiquetas = ["Pequeñas (10-49)", "Medianas (50-249)", "Grandes (250+)"]

# Posición de cada barrera
y = range(len(tabla_barreras_tamano))

# Línea que muestra el rango entre tamaños
for i, (_, fila) in enumerate(tabla_barreras_tamano.iterrows()):
    ax.plot(
        [fila[tamanos].min(), fila[tamanos].max()],
        [i, i],
        color="#CBD5E1",
        linewidth=3,
        zorder=1
    )

# Puntos de cada tamaño empresarial
for tamano, color, etiqueta in zip(tamanos, colores, etiquetas):
    ax.scatter(
        tabla_barreras_tamano[tamano],
        y,
        s=110,
        color=color,
        label=etiqueta,
        edgecolor="white",
        linewidth=1,
        zorder=3
    )

ax.set_yticks(list(y))
ax.set_yticklabels(tabla_barreras_tamano.index)
ax.invert_yaxis()

ax.set_title(
    "Barreras para adoptar IA según tamaño empresarial (2025)",
    fontsize=14,
    fontweight="bold",
    pad=20
)

ax.set_xlabel("Porcentaje (%)")
ax.set_ylabel("")
ax.set_xlim(0, 100)

ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.12),
    ncol=3,
    frameon=False
)

ax.grid(axis="x", alpha=0.2)
ax.set_axisbelow(True)

# Eliminar bordes innecesarios
sns.despine(left=True, bottom=True)

plt.tight_layout()
plt.show()

**Hallazgo.** La falta de conocimientos especializados es la barrera con mayor porcentaje (**81,29 %**). En la comparación por tamaño alcanza **83,21 %** en pequeñas y **74,18 %** en grandes. La interpretación se limita a la población de referencia del indicador, pendiente de confirmación metodológica.

## 6. H5 — Adopción de IA y productividad laboral por sector (2025)

La productividad laboral aparente se calcula como **VAB nominal (€) por persona ocupada**. Las cuentas nacionales de Eurostat incluyen el empleo del sector, mientras que la encuesta de adopción del INE cubre un universo empresarial más restringido. La comparación es exploratoria y no permite inferir causalidad.

In [ ]:
# Cargar los dos conjuntos de cuentas nacionales de Eurostat
df_vab = pd.read_csv(DATA_PATH / "vab_espana_2025.csv")
df_ocupados = pd.read_csv(DATA_PATH / "ocupados_espana_2025.csv")
print("VAB:", df_vab.shape, "| Empleo:", df_ocupados.shape)
print("Indicadores de empleo:", df_ocupados["na_item"].unique())

In [ ]:
# H5.4 - Productividad laboral por sector en España (2025)

# 1. Seleccionar el VAB
vab = df_vab[
    (df_vab["geo"] == "ES") &
    (df_vab["TIME_PERIOD"] == 2025) &
    (df_vab["na_item"] == "B1G") &
    (df_vab["unit"] == "CP_MEUR")
][["nace_r2", "OBS_VALUE"]].copy()

vab = vab.rename(columns={"OBS_VALUE": "vab_millones"})

# 2. Seleccionar el total de personas ocupadas
ocupados = df_ocupados[
    (df_ocupados["geo"] == "ES") &
    (df_ocupados["TIME_PERIOD"] == 2025) &
    (df_ocupados["na_item"] == "EMP_DC") &
    (df_ocupados["unit"] == "THS_PER")
][["nace_r2", "OBS_VALUE"]].copy()

ocupados = ocupados.rename(
    columns={"OBS_VALUE": "ocupados_miles"}
)

# 3. Unir ambos datasets por sector
df_productividad = pd.merge(
    vab,
    ocupados,
    on="nace_r2",
    how="inner",
    validate="one_to_one"
)

# 4. Calcular euros de VAB por persona ocupada
# Millones de euros / miles de personas × 1000
df_productividad["productividad_eur"] = (
    df_productividad["vab_millones"] /
    df_productividad["ocupados_miles"] * 1000
)

# 5. Ordenar y visualizar
df_productividad = df_productividad.sort_values(
    "productividad_eur",
    ascending=False
)

display(df_productividad.round(2))

In [ ]:
# H5.6 - Cruzar adopción de IA y productividad laboral

# Equivalencias INE - Eurostat
equivalencias = {
    "2. Total Construcción (CNAE 41-43)": "F",
    "3.2. Transporte y almacenamiento (CNAE 49-53)": "H",
    "3.4. Información y comunicaciones (CNAE 58-63)": "J",
    "3.5. Actividades inmobiliarias (CNAE 68)": "L",
    "3.6. Actividades profesionales, científicas y técnicas (CNAE 69-75)": "M"
}

# Preparar datos de adopción
df_ia_h5 = df_h3[
    df_h3["sector"].isin(equivalencias.keys())
][["sector", "value_pct"]].copy()

df_ia_h5["nace_r2"] = df_ia_h5["sector"].map(equivalencias)

df_ia_h5 = df_ia_h5.rename(
    columns={"value_pct": "adopcion_ia_pct"}
)

# Cruzar con productividad laboral
df_h5 = pd.merge(
    df_ia_h5,
    df_productividad[["nace_r2", "productividad_eur"]],
    on="nace_r2",
    how="inner",
    validate="one_to_one"
)

# Comprobar el resultado
print("Sectores comparados:", len(df_h5))

display(
    df_h5[
        ["sector", "adopcion_ia_pct", "productividad_eur"]
    ].round(2)
)

In [ ]:
# H5.7 - Correlación entre adopción de IA y productividad

# Correlación de Pearson con los 5 sectores
correlacion_total = df_h5[
    ["adopcion_ia_pct", "productividad_eur"]
].corr(method="pearson").iloc[0, 1]

# Correlación de Spearman con los 5 sectores
spearman_total = df_h5[
    ["adopcion_ia_pct", "productividad_eur"]
].corr(method="spearman").iloc[0, 1]

# Excluir actividades inmobiliarias para comprobar
# la influencia de su elevada productividad
df_h5_sin_inmobiliarias = df_h5[
    df_h5["nace_r2"] != "L"
].copy()

correlacion_sin_inmobiliarias = df_h5_sin_inmobiliarias[
    ["adopcion_ia_pct", "productividad_eur"]
].corr(method="pearson").iloc[0, 1]

print("CORRELACIONES H5")
print("------------------------------")
print(f"Pearson (5 sectores): {correlacion_total:.3f}")
print(f"Spearman (5 sectores): {spearman_total:.3f}")
print(
    f"Pearson sin inmobiliarias (4 sectores): "
    f"{correlacion_sin_inmobiliarias:.3f}"
)

In [ ]:
# H5.8 - Relación entre adopción de IA y productividad (2025)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

graficos = [
    (df_h5, "Todos los sectores (n=5)", correlacion_total),
    (
        df_h5_sin_inmobiliarias,
        "Sin actividades inmobiliarias (n=4)",
        correlacion_sin_inmobiliarias
    )
]

nombres = {
    "F": "Construcción",
    "H": "Transporte",
    "J": "Información y comunicaciones",
    "L": "Inmobiliarias",
    "M": "Actividades profesionales"
}

for ax, (datos, titulo, correlacion) in zip(axes, graficos):

    sns.regplot(
        data=datos,
        x="adopcion_ia_pct",
        y="productividad_eur",
        ax=ax,
        ci=None,
        scatter_kws={"s": 110, "color": "#2563EB"},
        line_kws={"color": "#F97316", "linewidth": 2}
    )

    for _, fila in datos.iterrows():
        ax.annotate(
            nombres[fila["nace_r2"]],
            (
                fila["adopcion_ia_pct"],
                fila["productividad_eur"]
            ),
            xytext=(6, 7),
            textcoords="offset points",
            fontsize=9
        )

    ax.set_title(
        f"{titulo}\nPearson r = {correlacion:.3f}",
        fontsize=12,
        fontweight="bold"
    )

    ax.set_xlabel("Empresas que utilizan IA (%)")
    ax.set_ylabel("Productividad laboral (€ por ocupado)")
    ax.grid(alpha=0.2)

fig.suptitle(
    "Adopción de IA y productividad laboral por sector en España (2025)",
    fontsize=15,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

**Hallazgo H5.** Con cinco sectores comparables, Pearson es **r = 0,124** y Spearman **ρ = 0,700**. Sin inmobiliarias, Pearson asciende a **0,935** con solo cuatro sectores: un resultado muy sensible a la selección de observaciones. **La hipótesis no queda confirmada de forma robusta**. El sector inmobiliario tiene un VAB por ocupado excepcionalmente alto, condicionado también por la estructura de capital y rentas del sector.

## 7. Conclusiones y limitaciones

La adopción de IA en España crece en el período analizado, pero presenta **brechas importantes por tamaño empresarial y sector económico**. Entre las barreras declaradas destaca la falta de conocimientos especializados. La asociación entre adopción sectorial de IA y productividad aparente **no es concluyente** con los datos disponibles.

**Limitaciones principales:** posibles cambios de definición entre 2022 y 2026; cambio de clasificación CNAE en 2026; indicadores de H4 con denominadores pendientes de confirmar; datos agregados sin seguimiento de empresas individuales; solo cinco sectores en H5; y cobertura diferente entre la encuesta del INE y las cuentas nacionales. Ninguna correlación demuestra causalidad.

**Reproducibilidad:** ejecutar las celdas en orden con los CSV originales en `src/data/raw/`. El archivo procesado se genera en `src/data/processed/`.